#  **Data Collection and Preprocessing**

### Import Libraries

In [1]:
import sys
import os

# Add parent directory to path so we can import from src
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, parent_dir)

In [2]:
from src.preprocessing import (
    clean_review_text,
    display_app_info,
    review_dataframe,
    remove_duplicates,
    handle_missing_data,
    normalize_dates,
    validate_rating,
    preprocessing_report,
    save_cleaned_data,
)
from src.data_scraping import scrap_reviews

### Web Scraping

#### App metadata

In [3]:
BOA_APP_ID = 'com.boa.boaMobileBanking'
display_app_info(BOA_APP_ID)

BoA Mobile App Info
App Title   : BoA Mobile
Current Score: 4.3978496
Total Ratings: 9,276
Total Reviews: 1,465
Installs     : 1,000,000+


#### Scrape reviews

In [4]:
reviews = scrap_reviews(app_id=BOA_APP_ID)

Scraping reviews for com.boa.boaMobileBanking...
Collected 700 raw reviews


#### Collect review text, rating, review date, bank , source

In [5]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(reviews[0].keys()))

print("\nFirst raw review (sample):")
for key, value in reviews[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 9dd3879e-2b0c-4058-8c8c-19f6b376f69c
  userName: Abel Legesse
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjVlAiPqwEucytjQHmhCbqlROOv44ryrgg5M6NZ2fb1qyojQN2u1
  content: The worst app, also bank am begging for my own money
  score: 1
  thumbsUpCount: 0
  reviewCreatedVersion: 26.05.11
  at: 2026-05-16 12:35:22
  replyContent: None
  repliedAt: None
  appVersion: 26.05.11


In [6]:
df = review_dataframe(reviews, app_info={'title': 'BOA Bank'})

print(f"Shape: {df.shape}")
df.head()

Shape: (700, 6)


,review_id,review,rating,date,bank,source
0,9dd3879e-2b0c-4058-8c8c-19f6b376f69c,"The worst app, also bank am begging for my own...",1,2026-05-16 12:35:22,BOA Bank,Google Play
1,5f69466d-ec06-4eb5-816c-296accffeff2,was Good 🙏,5,2026-05-16 00:10:06,BOA Bank,Google Play
2,f9246b8a-6688-4249-b804-0b0c5dd60590,cool,5,2026-05-15 21:07:21,BOA Bank,Google Play
3,21bcff26-5b05-485d-958f-4832ec1fac01,Its Good,5,2026-05-15 15:01:09,BOA Bank,Google Play
4,c5eb7589-59b7-4d72-8aa9-100a703ecaa3,good,5,2026-05-14 21:18:44,BOA Bank,Google Play


## Preprocessing

#### Remove duplicate reviews

In [7]:
df_clean = df.copy()

In [8]:
df_clean = remove_duplicates(df_clean)

Removed 0 duplicate reviews
Remaining: 700 reviews


#### Handle missing values

In [9]:
df_clean =handle_missing_data(df_clean)

Removed 0 rows with missing critical data
Remaining: 700 reviews


#### Normalize dates to YYYY-MM-DD format

In [10]:
df_clean = normalize_dates(df_clean)

Before normalization:
0   2026-05-16 12:35:22
1   2026-05-16 00:10:06
2   2026-05-15 21:07:21
dtype: datetime64[us]

After normalization:
0    2026-05-16
1    2026-05-16
2    2026-05-15
dtype: str

Date range: 2024-08-22 to 2026-05-16


#### Handle incorrect ratings

In [11]:
df_clean = validate_rating(df_clean)

All ratings are valid (1-5).
Remaining: 700 reviews


#### Clean review text

In [12]:
df['review'] = df['review'].apply(clean_review_text)

print("Sample cleaned reviews:")
print(df['review'].head(10).to_string())

Sample cleaned reviews:
0    the worst app, also bank am begging for my own...
1                                            was good 
2                                                 cool
3                                             its good
4                                                 good
5                                   it's very good app
6    this app is good but the speed of app is very ...
7                                                 good
8                                         boa the best
9           bank of absiniya is best bank in ethiopian


#### Save the cleaned dataset

In [13]:
# Select only the 5 required columns in the right order
df_clean = df_clean[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (700, 5)


,review,rating,date,bank,source
0,"The worst app, also bank am begging for my own...",1,2026-05-16,BOA Bank,Google Play
1,was Good 🙏,5,2026-05-16,BOA Bank,Google Play
2,cool,5,2026-05-15,BOA Bank,Google Play
3,Its Good,5,2026-05-15,BOA Bank,Google Play
4,good,5,2026-05-14,BOA Bank,Google Play
5,it's very good app,5,2026-05-12,BOA Bank,Google Play
6,this app is good but the speed of app is very ...,2,2026-05-11,BOA Bank,Google Play
7,good,5,2026-05-09,BOA Bank,Google Play
8,boa the best,5,2026-05-08,BOA Bank,Google Play
9,bank of absiniya is best bank in ethiopian,5,2026-05-07,BOA Bank,Google Play


In [14]:
save_cleaned_data(df_clean, output_path="../../data/processed/boa_reviews_cleaned.csv")

Cleaned data saved to ../../data/processed/boa_reviews_cleaned.csv
Saved to: ../../data/processed/boa_reviews_cleaned.csv


### Report

In [15]:
preprocessing_report(df, df_clean)

  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :    700
  Reviews after cleaning :    700
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2024-08-22  to  2026-05-16
Rating distribution:
  5 stars:  375  ███████████████████████████████████████████████████████████████████████████
  4 stars:   47  █████████
  3 stars:   35  ███████
  2 stars:   22  ████
  1 stars:  221  ████████████████████████████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 16 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

